# Supply Chain Early Warning System

This notebook builds a forward-looking supply chain risk monitor. Rather than reporting disruptions after the fact, it pulls live data from four external sources, scores risk across major global trade routes, and quantifies potential financial exposure before a stockout occurs.

The risk model uses a weighted composite score across five dimensions: geopolitical conflict, shipping costs, climate disruptions, trade policy, and port logistics performance.

**Risk formula:**
```
Risk Score = (Conflict x 0.30) + (Shipping Cost x 0.25) + (Climate x 0.20) + (Trade Policy x 0.15) + (Port Delay x 0.10)
```

**Thresholds:** Critical >= 75 | High >= 55 | Medium >= 35 | Low < 35

---

**Data sources:**
- Baltic Dry Index via HandyBulk (shipping cost proxy)
- World Bank Logistics Performance Index (port efficiency)
- NOAA Weather API (climate disruption)
- UN Comtrade API (trade policy risk)

---

**Setup:** Replace all `YOUR_*` placeholders in Cell 3 before running. You will need a Google Service Account JSON file for Sheets integration and a Gmail app password for email alerts.

In [ ]:
# Install required libraries
!pip install requests pandas numpy matplotlib seaborn gspread google-auth

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import smtplib
import json
import re
import time
import gspread
from google.oauth2.service_account import Credentials
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from datetime import datetime

In [ ]:
# Configuration
# Replace all YOUR_* values with your own before running

COMPANY_NAME   = "Your Company Name"
DAILY_REVENUE  = 850000

FRED_API_KEY      = "YOUR_FRED_API_KEY"
JSON_KEY_FILE     = "your-credentials.json"
SHEET_NAME        = "Your_Sheet_Name"
GMAIL_SENDER      = "your_email@gmail.com"
GMAIL_RECEIVER    = "your_email@gmail.com"
GMAIL_APP_PASSWORD = "YOUR_APP_PASSWORD"

# Product categories with daily revenue value and current stock cover in days
CATEGORIES = {
    "Electronics":   {"daily_value": 280000, "stock_days": 18},
    "Home & Garden": {"daily_value": 180000, "stock_days": 22},
    "Textiles":      {"daily_value": 150000, "stock_days": 25},
    "Furniture":     {"daily_value": 120000, "stock_days": 30},
    "Sports":        {"daily_value":  80000, "stock_days": 20},
    "Toys":          {"daily_value":  40000, "stock_days": 28},
}

# Trade routes with normal and disrupted transit times in days
ROUTES = {
    "Asia-Europe Red Sea":      {"normal_days": 28, "disrupted_days": 42},
    "Asia-Europe Suez":         {"normal_days": 25, "disrupted_days": 38},
    "Americas-Europe Panama":   {"normal_days": 20, "disrupted_days": 30},
    "Europe-Asia Trade Policy": {"normal_days": 22, "disrupted_days": 35},
}

RISK_THRESHOLDS = {"CRITICAL": 75, "HIGH": 55, "MEDIUM": 35, "LOW": 0}

In [ ]:
# Data pipeline — four live sources
# Each function returns a numeric risk score (0-100) or structured data
# Fallback values are used if an API is unavailable

def get_baltic_dry_index():
    """Baltic Dry Index scraped from HandyBulk. Used as a proxy for global shipping cost pressure."""
    try:
        url = "https://www.handybulk.com/baltic-dry-index/"
        response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
        match = re.search(r'to reach ([\d,]+) points', response.text)
        latest = int(match.group(1).replace(",", "")) if match else 2138
        if latest > 2000:   risk = 80
        elif latest > 1500: risk = 60
        elif latest > 1000: risk = 40
        else:               risk = 20
        print(f"Baltic Dry Index: {latest} — risk score: {risk}")
        return risk
    except Exception as e:
        print(f"BDI unavailable ({e}) — using fallback: 50")
        return 50


def get_world_bank_lpi():
    """World Bank Logistics Performance Index for key sourcing and destination countries."""
    try:
        url = "http://api.worldbank.org/v2/country/CN;VN;BD;DE;NL/indicator/LP.LPI.OVRL.XQ"
        response = requests.get(url, params={"format": "json", "mrv": 1}, timeout=15)
        data = response.json()
        scores = {}
        for item in data[1]:
            country = item["country"]["value"]
            score   = item["value"]
            if score:
                scores[country] = round(float(score), 2)
        print(f"World Bank LPI: {scores}")
        return scores
    except Exception as e:
        print(f"World Bank unavailable ({e}) — using fallback")
        return {"China": 3.7, "Vietnam": 3.3, "Bangladesh": 2.6, "Germany": 4.1, "Netherlands": 4.1}


def get_weather_risk():
    """Active weather alerts from NOAA. Alert count is used as a climate disruption proxy."""
    try:
        url = "https://api.weather.gov/alerts/active"
        response = requests.get(
            url,
            params={"status": "actual", "message_type": "alert"},
            headers={"User-Agent": "SupplyChainMonitor"},
            timeout=15
        )
        alert_count = len(response.json().get("features", []))
        risk = min(80, alert_count * 0.3)
        print(f"NOAA active alerts: {alert_count} — risk score: {risk:.0f}")
        return round(risk, 1)
    except Exception as e:
        print(f"NOAA unavailable ({e}) — using fallback: 30")
        return 30


def get_trade_policy_risk():
    """China to Germany export volume from UN Comtrade. High bilateral trade volume indicates elevated policy sensitivity."""
    try:
        url    = "https://comtradeapi.un.org/public/v1/preview/C/A/HS"
        params = {"reporterCode": "156", "partnerCode": "276",
                  "cmdCode": "TOTAL", "flowCode": "X", "period": "2022"}
        response    = requests.get(url, params=params, timeout=15)
        trade_value = response.json()["data"][0].get("primaryValue", 0)
        risk_score  = min(90, max(20, trade_value / 1e9 * 10))
        print(f"Comtrade China-Germany exports: ${trade_value/1e9:.1f}B — risk score: {risk_score:.0f}")
        return round(risk_score, 1)
    except Exception as e:
        print(f"Comtrade unavailable ({e}) — using fallback: 45")
        return 45


def get_conflict_risk():
    """
    Conflict risk scores by region based on current geopolitical conditions.
    These are manually verified and should be updated periodically.
    For live data, connect GDELT or ACLED once API access is approved.
    """
    conflict_risks = {
        "Red Sea":       90,
        "Taiwan Strait": 70,
        "Panama":        25,
        "Suez":          75,
    }
    for region, risk in conflict_risks.items():
        print(f"Conflict risk — {region}: {risk}")
    return conflict_risks


bdi_risk       = get_baltic_dry_index()
lpi_scores     = get_world_bank_lpi()
weather_risk   = get_weather_risk()
trade_risk     = get_trade_policy_risk()
conflict_risks = get_conflict_risk()

In [ ]:
# Risk scoring engine
# Calculates a weighted composite score for each route and assigns a risk level

def calculate_route_risk(route_name, conflict_risks, bdi_risk, weather_risk, trade_risk, lpi_scores):
    region_map = {
        "Asia-Europe Red Sea":      "Red Sea",
        "Asia-Europe Suez":         "Suez",
        "Americas-Europe Panama":   "Panama",
        "Europe-Asia Trade Policy": "Taiwan Strait"
    }

    region     = region_map[route_name]
    conflict   = conflict_risks.get(region, 50)
    avg_lpi    = sum(lpi_scores.values()) / len(lpi_scores)
    port_delay = round((5 - avg_lpi) / 5 * 100, 1)

    risk_score = (
        conflict     * 0.30 +
        bdi_risk     * 0.25 +
        weather_risk * 0.20 +
        trade_risk   * 0.15 +
        port_delay   * 0.10
    )
    risk_score = round(risk_score, 1)

    if risk_score >= 75:   level = "CRITICAL"
    elif risk_score >= 55: level = "HIGH"
    elif risk_score >= 35: level = "MEDIUM"
    else:                  level = "LOW"

    print(f"{level:<10} {route_name}: {risk_score}")
    return {"route": route_name, "risk_score": risk_score, "level": level}


print("Route risk scores\n" + "-" * 40)
all_scores = []
for route_name in ROUTES.keys():
    result = calculate_route_risk(route_name, conflict_risks, bdi_risk,
                                  weather_risk, trade_risk, lpi_scores)
    all_scores.append(result)

In [ ]:
# Stock cover analysis
# For each route, calculates how many days of stock remain per category
# if transit time increases to the disrupted scenario

def calculate_stock_cover(all_scores):
    stock_warnings = []

    for route_result in all_scores:
        route_name  = route_result["route"]
        extra_delay = ROUTES[route_name]["disrupted_days"] - ROUTES[route_name]["normal_days"]

        print(f"\n{route_name} (extra delay: {extra_delay} days)")

        for category, details in CATEGORIES.items():
            days_remaining = details["stock_days"] - extra_delay
            financial_risk = extra_delay * details["daily_value"]

            if days_remaining <= 0:    status = "STOCKOUT"
            elif days_remaining <= 5:  status = "CRITICAL"
            elif days_remaining <= 10: status = "WARNING"
            else:                      status = "SAFE"

            if days_remaining <= 10:
                print(f"  {status:<10} {category}: {days_remaining} days remaining — GBP {financial_risk:,.0f} at risk")
                stock_warnings.append({
                    "route": route_name,
                    "category": category,
                    "days_remaining": days_remaining,
                    "financial_risk": financial_risk
                })

    return stock_warnings


warnings = calculate_stock_cover(all_scores)

In [ ]:
# Financial exposure summary

def calculate_financial_exposure(warnings):
    total_exposure = 0
    route_exposure = {}

    for w in warnings:
        route = w["route"]
        route_exposure[route] = route_exposure.get(route, 0) + w["financial_risk"]
        total_exposure        += w["financial_risk"]

    print("Financial exposure by route\n" + "-" * 40)
    for route, exposure in route_exposure.items():
        print(f"  {route}: GBP {exposure:,.0f}")

    print(f"\nTotal exposure:  GBP {total_exposure:,.0f}")
    print(f"Daily revenue:   GBP {DAILY_REVENUE:,.0f}")
    print(f"Equivalent to {total_exposure / DAILY_REVENUE:.1f} days of revenue at risk")

    return total_exposure


total_exposure = calculate_financial_exposure(warnings)

In [ ]:
# Google Sheets integration and email alerts

def connect_to_sheets():
    scope  = ["https://spreadsheets.google.com/feeds",
              "https://www.googleapis.com/auth/drive"]
    creds  = Credentials.from_service_account_file(JSON_KEY_FILE, scopes=scope)
    client = gspread.authorize(creds)
    return client.open(SHEET_NAME).sheet1


def update_sheet(sheet, all_scores, warnings):
    """Appends new rows only. Checks existing data to prevent duplicates on the same date."""
    timestamp     = datetime.now().strftime("%Y-%m-%d %H:%M")
    today         = datetime.now().strftime("%Y-%m-%d")
    existing_data = sheet.get_all_values()

    if not existing_data:
        sheet.update([["Timestamp", "Route", "Risk Score", "Risk Level", "Financial Exposure (GBP)"]])
        existing_data = []

    existing_combos = set()
    for row in existing_data[1:]:
        if len(row) >= 2:
            existing_combos.add((row[0][:10], row[1]))

    new_rows = []
    for score in all_scores:
        route = score["route"]
        if (today, route) not in existing_combos:
            route_exposure = sum(w["financial_risk"] for w in warnings if w["route"] == route)
            new_rows.append([timestamp, route, score["risk_score"], score["level"], route_exposure])

    if new_rows:
        sheet.append_rows(new_rows)
        print(f"{len(new_rows)} rows written to sheet.")
    else:
        print("Sheet already up to date for today.")


def send_alert_email(critical_routes):
    body = f"Supply Chain Early Warning — {COMPANY_NAME}\n"
    body += f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n\n"
    body += "The following routes have reached CRITICAL risk level:\n\n"

    for route in critical_routes:
        route_exposure = sum(w["financial_risk"] for w in warnings if w["route"] == route["route"])
        body += f"  {route['route']}\n"
        body += f"  Risk score: {route['risk_score']}/100\n"
        body += f"  Financial exposure: GBP {route_exposure:,.0f}\n\n"

    body += f"Total company exposure: GBP {total_exposure:,.0f}\n"
    body += "Recommended action: review inventory levels and consider emergency restocking."

    msg            = MIMEMultipart()
    msg["From"]    = GMAIL_SENDER
    msg["To"]      = GMAIL_RECEIVER
    msg["Subject"] = f"Supply Chain Alert — CRITICAL risk detected — {COMPANY_NAME}"
    msg.attach(MIMEText(body, "plain"))

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(GMAIL_SENDER, GMAIL_APP_PASSWORD)
        server.sendmail(GMAIL_SENDER, GMAIL_RECEIVER, msg.as_string())
    print(f"Alert sent to {GMAIL_RECEIVER}")


# Run
sheet = connect_to_sheets()
update_sheet(sheet, all_scores, warnings)

critical_routes = [s for s in all_scores if s["level"] == "CRITICAL"]
if critical_routes:
    send_alert_email(critical_routes)
else:
    print("No critical routes detected.")

In [ ]:
# Visualisations

def create_visualizations(all_scores, warnings):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'Supply Chain Risk Dashboard — {COMPANY_NAME}', fontsize=13, fontweight='bold')

    # Route risk scores
    routes = [s["route"].replace("Asia-Europe ", "A-E ").replace("Americas-Europe ", "Am-E ").replace("Europe-Asia ", "E-A ") for s in all_scores]
    scores = [s["risk_score"] for s in all_scores]
    colors = ["#c62828" if s >= 75 else "#ef6c00" if s >= 55 else "#f9a825" if s >= 35 else "#2e7d32" for s in scores]
    axes[0, 0].barh(routes, scores, color=colors)
    axes[0, 0].set_title('Route Risk Scores')
    axes[0, 0].set_xlabel('Risk Score (0-100)')
    axes[0, 0].axvline(x=75, color='#c62828', linestyle='--', alpha=0.4, label='Critical threshold')
    axes[0, 0].axvline(x=55, color='#ef6c00', linestyle='--', alpha=0.4, label='High threshold')
    axes[0, 0].legend(fontsize=8)

    # Financial exposure by route
    route_exp = {}
    for w in warnings:
        r = w["route"].replace("Asia-Europe ", "A-E ").replace("Americas-Europe ", "Am-E ").replace("Europe-Asia ", "E-A ")
        route_exp[r] = route_exp.get(r, 0) + w["financial_risk"]
    axes[0, 1].bar(route_exp.keys(), [v / 1e6 for v in route_exp.values()], color='#b71c1c')
    axes[0, 1].set_title('Financial Exposure by Route')
    axes[0, 1].set_ylabel('Exposure (GBP millions)')
    axes[0, 1].tick_params(axis='x', rotation=15)

    # Minimum stock days remaining by category
    if warnings:
        warn_df    = pd.DataFrame(warnings)
        warn_df    = warn_df.groupby('category')['days_remaining'].min().reset_index()
        bar_colors = ["#c62828" if d <= 0 else "#ef6c00" if d <= 5 else "#f9a825" for d in warn_df['days_remaining']]
        axes[1, 0].bar(warn_df['category'], warn_df['days_remaining'], color=bar_colors)
        axes[1, 0].set_title('Minimum Stock Days Remaining (Worst Case)')
        axes[1, 0].set_ylabel('Days')
        axes[1, 0].axhline(y=5, color='#c62828', linestyle='--', alpha=0.4, label='Critical threshold')
        axes[1, 0].legend(fontsize=8)
        axes[1, 0].tick_params(axis='x', rotation=15)

    # Risk contribution by data source
    avg_lpi       = sum(lpi_scores.values()) / len(lpi_scores)
    port_delay    = round((5 - avg_lpi) / 5 * 100, 1)
    avg_conflict  = sum(conflict_risks.values()) / len(conflict_risks)
    sources       = ['Conflict\n(30%)', 'Shipping\n(25%)', 'Climate\n(20%)', 'Trade\nPolicy (15%)', 'Port\nDelay (10%)']
    contributions = [
        avg_conflict * 0.30,
        bdi_risk     * 0.25,
        weather_risk * 0.20,
        trade_risk   * 0.15,
        port_delay   * 0.10
    ]
    axes[1, 1].bar(sources, contributions, color=['#c62828', '#ef6c00', '#1565c0', '#6a1b9a', '#2e7d32'])
    axes[1, 1].set_title('Risk Contribution by Data Source')
    axes[1, 1].set_ylabel('Weighted points')

    plt.tight_layout()
    plt.savefig('supply_chain_risk.png', dpi=150, bbox_inches='tight')
    plt.show()


create_visualizations(all_scores, warnings)